<a href="https://colab.research.google.com/github/gauravd12345/miniCLIP/blob/main/miniCLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eeshawn/flickr30k")

print("Path to dataset files:", path)

images = path + '/flickr30k_images'
captions = path + '/captions.txt'

Using Colab cache for faster access to the 'flickr30k' dataset.
Path to dataset files: /kaggle/input/flickr30k


In [99]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
from PIL import Image
from IPython.display import display

In [100]:
""" Vision Transformer parameters """
H = 128               # image height
W = 128               # image width
C = 3                 # number of channels

P = 32                # patch resolution
N = (H * W) // P**2   # number of patches

batch_size = 64

""" Transformer parameters """
d_model = 512
d_k = 64
d_v = 64
h = 8
N_t = 6

In [101]:
import pandas as pd

df = pd.read_csv(captions)

image_names = df['image_name']
comment_numbers = df['comment_number']
comments = df['comment']

In [102]:
class ViTDataset(Dataset):
    def __init__(self, image_names, captions, transform=None):
        self.captions = captions
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = Image.open(f"{images}/{self.image_names[idx]}")
        if self.transform:
            img = self.transform(img)         # extract H, W, C values
        img = img.reshape(N, P**2 * C)        # flatten
        return img, self.captions[idx]


transform = transforms.Compose([ # resizing images
    transforms.Resize((H, W)),
    transforms.ToTensor()
])

dataset = ViTDataset(image_names, comments, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

for img, caption in dataloader:
    print(f"Samples per batch: {len(dataloader)}")
    print(f"Image batch shape: {img.shape}")
    print(f"Number of captions: {len(caption)}")
    break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Samples per batch: 2484
Image batch shape: torch.Size([64, 16, 3072])
Number of captions: 64
